# 2 — Email Triage
> An agent that reads raw email text and returns urgency, category, a one-line summary, and a recommended action — every time, in a consistent format your systems can act on.

*Run all cells. Swap the emails in the final code cell with your own.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import Literal
from pydantic import BaseModel
from langchain_openai import ChatOpenAI


class EmailTriage(BaseModel):
    urgency: Literal["high", "medium", "low"]
    category: Literal["billing", "technical", "general", "spam"]
    summary: str
    recommended_action: str


def run(emails: list[tuple[str, str]]) -> list[dict]:
    """Classify a list of (label, email_text) pairs. Returns a list of result dicts."""
    classifier = ChatOpenAI(model="gpt-4o-mini").with_structured_output(EmailTriage)
    results = []
    for label, text in emails:
        result = classifier.invoke(text)
        results.append({
            "label": label,
            "urgency": result.urgency,
            "category": result.category,
            "summary": result.summary,
            "recommended_action": result.recommended_action,
        })
    return results

## The scenario

It's Monday morning. Five emails arrived overnight — a payment failure from a key account, a security alert, a churn risk from an unhappy customer, a routine renewal, and a cold pitch from a vendor. The agent reads each one and tells you exactly what it is, how urgent it is, and what to do next.

In [ ]:
EMAILS = [
    (
        "Payment failure",
        """Subject: Payment failed - subscription at risk
From: billing@stripe.com

We were unable to charge $899.00 to the card on file for Meridian Analytics (customer ID: cus_X9k2).
This is the second failed attempt. If payment is not resolved within 48 hours, the account
will be downgraded to the free tier and all active workflows will be paused.""",
    ),
    (
        "Security alert",
        """Subject: Unusual login detected on your admin account
From: security@yourdomain.com

We detected a login to the admin panel from an unrecognized IP address (213.44.17.92, Romania)
at 03:14 UTC. Two-factor authentication was not used. If this was not you, your credentials
may be compromised. Please rotate your password and review recent activity immediately.""",
    ),
    (
        "Churn risk",
        """Subject: Reconsidering our subscription
From: sarah.okonkwo@brightfield.co

Hi, I'm the operations lead at Brightfield. We've been on the Pro plan for eight months
but honestly we're not getting the value we expected. The reporting exports are still broken
and support response times have slipped. We're reviewing our tools this week and I wanted
to give you a chance to respond before we make a decision.""",
    ),
    (
        "Renewal confirmation",
        """Subject: Your annual plan has been renewed
From: noreply@yourdomain.com

This is a confirmation that the annual Pro plan for Hartwell Group has been successfully
renewed for $2,400. Your next renewal date is June 18, 2027. No action is required.
A receipt has been sent to finance@hartwellgroup.com.""",
    ),
    (
        "Vendor cold pitch",
        """Subject: Cut your cloud costs by 40% — 15-minute call?
From: tom@cloudoptimize.io

Hey, I help SaaS companies reduce AWS spend by an average of 40% in 30 days. I took a look
at your public stack and think there's real opportunity. Would you be open to a quick call
this week? Happy to send over a free audit first. Let me know!""",
    ),
]

results = run(EMAILS)

URGENCY_ICON = {"high": "[!!!]", "medium": "[ ! ]", "low": "[   ]"}

print(f"{'':6} {'EMAIL':<22} {'URGENCY':<10} {'CATEGORY':<14} {'SUMMARY'}")
print("-" * 95)
for r in results:
    icon = URGENCY_ICON[r["urgency"]]
    print(f"{icon}  {r['label']:<22} {r['urgency']:<10} {r['category']:<14} {r['summary'][:45]}")

print()
print("RECOMMENDED ACTIONS")
print("-" * 95)
for r in results:
    print(f"  {r['label']:<22}  {r['recommended_action']}")

## Use your own data

Replace the `EMAILS` list above with:
- Your own email subjects, senders, and body text (paste raw email content as strings)
- As many or as few emails as you want — one at a time or in bulk

The agent will return urgency (`high` / `medium` / `low`), category (`billing` / `technical` / `general` / `spam`), a one-line summary, and a concrete recommended action for each one.